# mIF Pipeline Debug Notebook

This notebook is a notebook-first wrapper around the Python API in `src/mif_pipeline`.

Use it to:
- inspect resolved config paths
- dry-run the full slide plan
- run one stage at a time and rerun only the failing stage
- inspect outputs between steps

Real data paths are expected to be cluster paths, so the dry-run cells are useful locally.

Set `REPO_ROOT` explicitly in the next code cell before running the rest of the notebook.

Important setup note: `setup_slide(...)` writes a starter channel map to `setup.channel_map_output`. If you want downstream steps in this notebook session to use that generated file immediately, flip `USE_GENERATED_CHANNEL_MAP = True` below and rerun the config-loading cell after the setup cell has written the file.


In [ ]:
# Set this to your checkout of the repo before running the notebook.
REPO_ROOT = "/home/ratnayn/Analysis/M10_admix/mIF-03262026/mIF-pipeline"


In [ ]:
from pathlib import Path
import json
import sys
import traceback

if "REPO_ROOT" not in globals():
    raise RuntimeError(
        "Set REPO_ROOT to the repository path before running this cell. Example: REPO_ROOT = '/abs/path/to/mIF-pipeline'"
    )

REPO_ROOT = Path(REPO_ROOT).expanduser().resolve()
if not (REPO_ROOT / "src").exists():
    raise RuntimeError(
        f"REPO_ROOT does not look like the repo root: {REPO_ROOT}. Expected to find {REPO_ROOT / 'src'}."
    )

SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"Repo root: {REPO_ROOT}")
print(f"Source dir added to sys.path: {SRC_DIR}")


In [ ]:
from mif_pipeline import (
    export_masks,
    load_config,
    merge_slide_ometiffs,
    qc_slide,
    run_all,
    run_instanseg,
    run_nimbus_chunked,
    setup_slide,
)
from mif_pipeline.config import get_slide_config, resolve_channel_entries, resolve_nimbus_inputs
from mif_pipeline.export_masks import mask_stems_for_slide
from mif_pipeline.instanseg_runner import instanseg_zarr_path


def show(data):
    print(json.dumps(data, indent=2, default=str))


def stage_call(label, func, *args, **kwargs):
    print(f"=== {label} ===")
    try:
        result = func(*args, **kwargs)
        show(result)
        return result
    except Exception as exc:
        print(f"{label} failed: {exc}")
        traceback.print_exc()
        raise


In [ ]:
CONFIG_PATH = REPO_ROOT / "example.yaml"
SLIDE_ID = "SLIDE-0272"

FORCE = False
RUN_DRY = True

# If setup_slide writes a starter map and you want this notebook session
# to use it immediately, flip this to True and rerun the next cell.
USE_GENERATED_CHANNEL_MAP = False

print(f"CONFIG_PATH = {CONFIG_PATH}")
print(f"SLIDE_ID = {SLIDE_ID}")
print(f"FORCE = {FORCE}")
print(f"RUN_DRY = {RUN_DRY}")
print(f"USE_GENERATED_CHANNEL_MAP = {USE_GENERATED_CHANNEL_MAP}")


In [ ]:
config = load_config(CONFIG_PATH)
slide = get_slide_config(config, SLIDE_ID)

if USE_GENERATED_CHANNEL_MAP:
    generated_map = slide.get("setup", {}).get("channel_map_output")
    if not generated_map:
        raise ValueError(
            "USE_GENERATED_CHANNEL_MAP is True but setup.channel_map_output is missing."
        )
    config["slides"][SLIDE_ID]["channel_map_file"] = generated_map
    slide = get_slide_config(config, SLIDE_ID)

nimbus_inputs = resolve_nimbus_inputs(config, SLIDE_ID)
seg_entries = resolve_channel_entries(
    config, SLIDE_ID, slide.get("seg_merge", {}).get("channels", [])
)
full_entries = resolve_channel_entries(
    config, SLIDE_ID, slide.get("full_merge", {}).get("channels", [])
)
mask_stems = mask_stems_for_slide(config, SLIDE_ID, image_paths=nimbus_inputs["fov_paths"])

ome_path = Path(slide["seg_merge"]["ome_path"])
zarr_path = instanseg_zarr_path(ome_path, slide["instanseg"]["prediction_tag"])

summary = {
    "slide_id": SLIDE_ID,
    "slide_dir": slide["slide_dir"],
    "output_dir": slide["output_dir"],
    "channel_map_file": slide["channel_map_file"],
    "setup_channel_map_output": slide.get("setup", {}).get("channel_map_output"),
    "seg_merge_inputs": [entry["path"] for entry in seg_entries],
    "full_merge_inputs": [entry["path"] for entry in full_entries],
    "nimbus_raw_paths": nimbus_inputs["raw_paths"],
    "nimbus_fov_paths": nimbus_inputs["fov_paths"],
    "mask_stems": mask_stems,
    "segmentation_ome_path": str(ome_path),
    "instanseg_zarr_path": str(zarr_path),
}
show(summary)


In [ ]:
def import_status(module_name):
    try:
        __import__(module_name)
        return "OK"
    except Exception as exc:
        return f"MISSING: {exc}"


dependency_status = {
    "yaml": import_status("yaml"),
    "tifffile": import_status("tifffile"),
    "ome_types": import_status("ome_types"),
    "numpy": import_status("numpy"),
    "skimage": import_status("skimage"),
    "zarr": import_status("zarr"),
    "instanseg": import_status("instanseg"),
    "tiffslide": import_status("tiffslide"),
    "nimbus_inference": import_status("nimbus_inference"),
    "pandas": import_status("pandas"),
}
show(dependency_status)


## Dry-Run Planning

Leave `RUN_DRY = True` while you validate config resolution, output paths, and chunk planning.


In [ ]:
dry_run_plan = stage_call("run_all(dry_run=True)", run_all, config, SLIDE_ID, dry_run=True)


## Run Individual Stages

Set `RUN_DRY = False` above when you are on the cluster and ready to execute a real step.

These are split into separate cells so you can rerun only the stage that failed.


In [ ]:
if slide.get("setup"):
    setup_result = stage_call(
        f"setup_slide(dry_run={RUN_DRY})",
        setup_slide,
        config,
        SLIDE_ID,
        force=FORCE,
        dry_run=RUN_DRY,
    )
else:
    print("No setup block configured for this slide.")


In [ ]:
merge_result = stage_call(
    f"merge_slide_ometiffs(dry_run={RUN_DRY})",
    merge_slide_ometiffs,
    config,
    SLIDE_ID,
    force=FORCE,
    dry_run=RUN_DRY,
)


In [ ]:
instanseg_result = stage_call(
    f"run_instanseg(dry_run={RUN_DRY})",
    run_instanseg,
    config,
    SLIDE_ID,
    force=FORCE,
    dry_run=RUN_DRY,
)


In [ ]:
export_result = stage_call(
    f"export_masks(dry_run={RUN_DRY})",
    export_masks,
    config,
    SLIDE_ID,
    image_paths=nimbus_inputs["fov_paths"],
    force=FORCE,
    dry_run=RUN_DRY,
)


In [ ]:
nimbus_result = stage_call(
    f"run_nimbus_chunked(dry_run={RUN_DRY})",
    run_nimbus_chunked,
    config,
    SLIDE_ID,
    force=FORCE,
    dry_run=RUN_DRY,
)


In [ ]:
qc_result = stage_call("qc_slide", qc_slide, config, SLIDE_ID)


## Full Run Shortcut

After debugging individual stages, you can use the single `run_all(...)` wrapper below.


In [ ]:
run_all_result = stage_call(
    f"run_all(dry_run={RUN_DRY})",
    run_all,
    config,
    SLIDE_ID,
    force=FORCE,
    dry_run=RUN_DRY,
)
